# WESAD

In [18]:
import os, sys

import numpy as np
import pandas as pd

from tqdm import tqdm

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
print(notebook_dir)
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

/Users/detraviousjamaribrinkley/Documents/Development/self/applied_time_series_and_machine_learning/misc/play


In [14]:
df = pd.read_csv("../datasets/WESAD/S2/S2_E4_Data/HR.csv")
df

,1495437335.000000
0,1.00
1,101.00
2,106.50
3,95.33
4,90.25
...,...
7861,81.38
7862,82.03
7863,82.67
7864,83.35


In [17]:
df = pd.read_pickle("../datasets/WESAD/S2/S2.pkl")
df

{'signal': {'chest': {'ACC': array([[ 0.95539999, -0.222     , -0.55799997],
          [ 0.92579997, -0.2216    , -0.55379999],
          [ 0.90820003, -0.21960002, -0.53920001],
          ...,
          [ 0.87179995, -0.12379998, -0.30419999],
          [ 0.87300003, -0.12339997, -0.30260003],
          [ 0.87020004, -0.12199998, -0.30220002]]),
   'ECG': array([[ 0.02142334],
          [ 0.02032471],
          [ 0.01652527],
          ...,
          [-0.00544739],
          [ 0.00013733],
          [ 0.0040741 ]]),
   'EMG': array([[-0.00444031],
          [ 0.00434875],
          [ 0.00517273],
          ...,
          [-0.01716614],
          [-0.02897644],
          [-0.02357483]]),
   'EDA': array([[5.25054932],
          [5.26733398],
          [5.24330139],
          ...,
          [0.36048889],
          [0.36582947],
          [0.365448  ]]),
   'Temp': array([[30.120758],
          [30.129517],
          [30.138214],
          ...,
          [31.459229],
          [31.484283

In [22]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import mode

# === Config ===
fs = 4  # wrist EDA sample rate in Hz
step = fs * 1  # 1-second stride
subject_ids = [f"S{i}" for i in range(2, 18) if i != 12]
base_path = "../datasets/WESAD"

# === Simulate cortisol ===
def simulate_cortisol_from_eda(eda_norm, labels, sid=None):
    eda_smooth = np.convolve(eda_norm, np.ones(20) / 20, mode='same')

    min_len = min(len(eda_smooth), len(labels))

    # === Debug: Inspect labels for stress ===
    print(f"\n🔍 First 100 labels for {sid}:")
    print(labels[:100])

    stress_indices = np.where(labels == 2)[0]
    print(f"{sid} → Found {len(stress_indices)} stress samples (label == 2)")

    if len(stress_indices) > 0:
        print(f"{sid} → First stress at index {stress_indices[0]} → time = {stress_indices[0]/fs:.2f} sec = {stress_indices[0]/fs/60:.2f} min")
        print(f"Next few: {stress_indices[:10]}")
    else:
        print(f"{sid} → ⚠️ No stress samples found!")

    eda_smooth = eda_smooth[:min_len]
    labels = labels[:min_len]
    eda_norm = eda_norm[:min_len]

    stress_signal = (labels == 2).astype(float)
    print(f"{sid} → {np.sum(stress_signal)} stress events ({100*np.mean(stress_signal):.1f}%)")

    cortisol = np.zeros_like(stress_signal)
    decay = 0.995
    for i in range(1, len(cortisol)):
        cortisol[i] = cortisol[i-1] * decay + stress_signal[i] * (eda_smooth[i] * 8 + 2.0)

    if np.max(cortisol) > 0:
        cortisol = cortisol / np.max(cortisol)

    return cortisol, labels, eda_smooth

# === Extract stats ===
def extract_hpa_dynamics(cortisol, time_curve):
    peak = np.max(cortisol)
    baseline = cortisol[0]
    delta = peak - baseline
    auc = np.trapz(cortisol, time_curve)
    peak_idx = np.argmax(cortisol)

    recovery_time = None
    for i in range(peak_idx, len(cortisol)):
        if cortisol[i] <= baseline + 0.1 * delta:
            recovery_time = time_curve[i]
            break

    return {
        "peak": peak,
        "baseline": baseline,
        "delta": delta,
        "auc": auc,
        "recovery_time": recovery_time,
        "time_to_peak": time_curve[peak_idx]
    }

# === Main ===
summary_stats = []

for sid in subject_ids:

    path = os.path.join(base_path, sid, f"{sid}.pkl")
    if not os.path.exists(path):
        print(f"❌ Missing: {sid}")
        continue

    with open(path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    eda = np.ravel(data["signal"]["wrist"]["EDA"])
    full_labels = np.ravel(data["label"])
    print(f"✅ Raw label distribution for {sid}: {dict(zip(*np.unique(full_labels, return_counts=True)))}")

    # Trim EDA to match downsampling step
    trim_len = len(eda) - (len(eda) % step)
    eda = eda[:trim_len]

    # Downsample EDA
    eda_down = np.mean(eda.reshape(-1, step), axis=1)

    # Downsample labels (700 Hz → 1 Hz)
    label_fs = 700
    label_step = label_fs // fs  # 700 / 4 = 175
    label_trim_len = len(full_labels) - (len(full_labels) % label_step)
    labels_down = full_labels[:label_trim_len].reshape(-1, label_step)

    # Custom downsampling: Prioritize stress label (2) if present in the window
    labels = np.zeros(len(labels_down), dtype=int)
    for i in range(len(labels_down)):
        window = labels_down[i]
        if 2 in window:  # If stress label exists, prioritize it
            labels[i] = 2
        else:
            labels[i] = mode(window, keepdims=False)[0]  # Otherwise, take the mode

    # Align lengths
    min_len = min(len(eda_down), len(labels))
    eda_down = eda_down[:min_len]
    labels = labels[:min_len]

    print(f"{sid} → Label distribution: {dict(zip(*np.unique(labels, return_counts=True)))}")
    print(f"{sid} → {np.sum(labels == 2)} samples labeled as 'stress'")

    # Normalize EDA
    eda_norm = eda_down / np.max(eda_down)
    duration_hrs = len(eda_norm) / fs / 3600
    time_axis = np.linspace(0, duration_hrs, len(eda_norm))

    cortisol, labels, eda_smooth = simulate_cortisol_from_eda(eda_norm, labels, sid=sid)
    metrics = extract_hpa_dynamics(cortisol, time_axis)
    metrics["subject"] = sid
    summary_stats.append(metrics)

    # === Plot ===
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(time_axis, cortisol, label="Simulated Cortisol", color="green")
    ax.plot(time_axis, eda_norm, label="EDA (normalized)", color="purple", alpha=0.4)
    ax.fill_between(time_axis, 0, 0.1 + 0.1 * (labels == 2), color='red', alpha=0.1, label="Label = 2")
    ax.set_title(f"{sid} — HPA Dynamics")
    ax.set_xlabel("Time (hours)")
    ax.set_ylabel("Level")
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"hpa_dynamics_{sid}.png")
    plt.close()

    print(f"✅ Saved: hpa_dynamics_{sid}.png")

# === Save summary ===
df = pd.DataFrame(summary_stats)
# df.to_csv("hpa_dynamics_summary.csv", index=False)
print("📊 Summary saved to hpa_dynamics_summary.csv")

✅ Raw label distribution for S2: {np.int32(0): np.int64(2142701), np.int32(1): np.int64(800800), np.int32(2): np.int64(430500), np.int32(3): np.int64(253400), np.int32(4): np.int64(537599), np.int32(6): np.int64(45500), np.int32(7): np.int64(44800)}
S2 → Label distribution: {np.int64(0): np.int64(1503), np.int64(1): np.int64(4576)}
S2 → 0 samples labeled as 'stress'

🔍 First 100 labels for S2:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
S2 → Found 0 stress samples (label == 2)
S2 → ⚠️ No stress samples found!
S2 → 0.0 stress events (0.0%)
✅ Saved: hpa_dynamics_S2.png


/var/folders/78/9z0b45fx1xqbwxh8vk97lcfh0000gn/T/ipykernel_10110/539972269.py:55: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = np.trapz(cortisol, time_curve)


✅ Raw label distribution for S3: {np.int32(0): np.int64(2345699), np.int32(1): np.int64(798000), np.int32(2): np.int64(448000), np.int32(3): np.int64(262500), np.int32(4): np.int64(546001), np.int32(5): np.int64(51100), np.int32(6): np.int64(46900), np.int32(7): np.int64(46900)}
S3 → Label distribution: {np.int64(0): np.int64(1641), np.int64(1): np.int64(4560), np.int64(5): np.int64(292)}
S3 → 0 samples labeled as 'stress'

🔍 First 100 labels for S3:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
S3 → Found 0 stress samples (label == 2)
S3 → ⚠️ No stress samples found!
S3 → 0.0 stress events (0.0%)
✅ Saved: hpa_dynamics_S3.png
✅ Raw label distribution for S4: {np.int32(0): np.int64(2314199), np.int32(1): np.int64(810601), np.int32(2): np.int64(444500), np.int32(3): np.int64(260400), np.int32(4): np.int64(563500), np.int32(5): np.in